In [67]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.anova import anova_lm

import statsmodels.formula.api as smf

In [69]:
og_summary = pd.read_parquet(
    "/scratch/suyuelyu/deimm/data/oma/oma_probe_val_test_og_summary.parquet"
)

In [ ]:
def get_correct(pred_rank_per_pos, sample_ranks):
    return [pred == sample_ranks[-1] for pred in pred_rank_per_pos]

In [99]:
with open(
    "/scratch/suyuelyu/deimm/results/probe_taxon/per_pos_eval/probe_taxon_per_pos_test_results_phylum.pkl",
    "rb",
) as f:
    phylum_per_pos = pickle.load(f)
phylum_per_pos = pd.DataFrame(phylum_per_pos)

phylum_per_pos["correct"] = phylum_per_pos.apply(
    lambda row: get_correct(row["pred_tgt_rank_id"], row["rank_ids"]), axis=1
)
phylum_per_pos["mean_accuracy"] = phylum_per_pos["correct"].apply(
    lambda x: sum(x) / len(x)
)
phylum_per_pos["n_src_proteins"] = phylum_per_pos["protein_names"].apply(
    lambda x: len(x) - 1 if x is not None else 0
)
phylum_per_pos['single_or_multi_rank'] = phylum_per_pos['rank_ids'].apply(lambda x: 'single' if len(set(x)) == 1 else 'multi') 
og_to_protein = dict(
    zip(phylum_per_pos["og"], phylum_per_pos["protein_names"].apply(lambda x: x[0]))
)


In [101]:
len(phylum_per_pos['og'].value_counts())

20

In [96]:
multi_1134260 = phylum_per_pos[
    (phylum_per_pos["og"] == "1134260") & (phylum_per_pos["single_or_multi_rank"] == "multi")
]
multi_1134261 = phylum_per_pos[
    (phylum_per_pos["og"] == "1134261") & (phylum_per_pos["single_or_multi_rank"] == "multi")
]
multi_1136248 = phylum_per_pos[
    (phylum_per_pos["og"] == "1136248") & (phylum_per_pos["single_or_multi_rank"] == "multi")
]

In [95]:
# phylum_per_pos[phylum_per_pos["tgt_rank_id"] == 5878.0] get mean when single vs multi rank
phylum_per_pos[phylum_per_pos["tgt_rank_id"] == 5878.0].groupby("single_or_multi_rank")[
    "mean_accuracy"
].mean()

single_or_multi_rank
multi     0.337874
single    0.531130
Name: mean_accuracy, dtype: float64

In [78]:
og_1134261 = phylum_per_pos[phylum_per_pos["og"] == "1134261"]

In [80]:
model = smf.ols(
    "mean_accuracy ~ n_src_proteins + C(src_rank_id) + C(tgt_rank_id) + C(single_or_multi_rank)",
    data=og_1134261,
).fit()
print(model.summary())

print(anova_lm(model, typ=2))

                            OLS Regression Results                            
Dep. Variable:          mean_accuracy   R-squared:                       0.736
Model:                            OLS   Adj. R-squared:                  0.734
Method:                 Least Squares   F-statistic:                     271.4
Date:                Mon, 09 Mar 2026   Prob (F-statistic):               0.00
Time:                        12:14:57   Log-Likelihood:                 5557.6
No. Observations:                3930   AIC:                        -1.103e+04
Df Residuals:                    3889   BIC:                        -1.078e+04
Df Model:                          40                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [97]:
model = smf.ols(
    "mean_accuracy ~ n_src_proteins + C(src_rank_id) + C(tgt_rank_id)",
    data=multi_1136248,
).fit()
print(model.summary())

print(anova_lm(model, typ=2))

                            OLS Regression Results                            
Dep. Variable:          mean_accuracy   R-squared:                       0.616
Model:                            OLS   Adj. R-squared:                  0.614
Method:                 Least Squares   F-statistic:                     283.6
Date:                Mon, 09 Mar 2026   Prob (F-statistic):               0.00
Time:                        12:29:14   Log-Likelihood:                 20590.
No. Observations:               12600   AIC:                        -4.104e+04
Df Residuals:                   12528   BIC:                        -4.050e+04
Df Model:                          71                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [74]:
model = smf.ols(
    "mean_accuracy ~ n_src_proteins + C(src_rank_id) + C(tgt_rank_id)",
    data=multi_1134261,
).fit()
print(model.summary())

print(anova_lm(model, typ=2))

                            OLS Regression Results                            
Dep. Variable:          mean_accuracy   R-squared:                       0.789
Model:                            OLS   Adj. R-squared:                  0.787
Method:                 Least Squares   F-statistic:                     361.1
Date:                Mon, 09 Mar 2026   Prob (F-statistic):               0.00
Time:                        12:01:39   Log-Likelihood:                 6116.5
No. Observations:                3800   AIC:                        -1.215e+04
Df Residuals:                    3760   BIC:                        -1.190e+04
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [68]:
model = smf.ols(
    "mean_accuracy ~ n_src_proteins + C(src_rank_id) + C(tgt_rank_id)",
    data=multi_1134260,
).fit()
print(model.summary())

print(anova_lm(model, typ=2))

                            OLS Regression Results                            
Dep. Variable:          mean_accuracy   R-squared:                       0.635
Model:                            OLS   Adj. R-squared:                  0.632
Method:                 Least Squares   F-statistic:                     228.6
Date:                Mon, 09 Mar 2026   Prob (F-statistic):               0.00
Time:                        11:52:32   Log-Likelihood:                 10178.
No. Observations:                7020   AIC:                        -2.025e+04
Df Residuals:                    6966   BIC:                        -1.988e+04
Df Model:                          53                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [91]:

# mean_accuracy ~ n_src_proteins + categorical og
model = smf.ols("mean_accuracy ~ n_src_proteins + C(og) + C(single_or_multi_rank) + C(tgt_rank_id)", data=phylum_per_pos).fit()
print(model.summary())
# significance of og as a whole (after controlling for n_src_proteins)
print(anova_lm(model, typ=2))
# model = smf.mixedlm(
#     "mean_accuracy ~ n_src_proteins + C(single_or_multi_rank) + C(tgt_rank_id)",
#     data=phylum_per_pos,
#     groups=phylum_per_pos["og"],
# ).fit()
# print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          mean_accuracy   R-squared:                       0.636
Model:                            OLS   Adj. R-squared:                  0.635
Method:                 Least Squares   F-statistic:                     627.2
Date:                Mon, 09 Mar 2026   Prob (F-statistic):               0.00
Time:                        12:20:05   Log-Likelihood:                 14496.
No. Observations:               11140   AIC:                        -2.893e+04
Df Residuals:                   11108   BIC:                        -2.869e+04
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

/scratch/suyuelyu/deimm/deimm/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 14, but rank is 3
  warnings.warn('covariance of constraints does not have full '
